<a href="https://colab.research.google.com/github/Decuayer/ce477-data-science-term-project/blob/feature%2Fdata-pipeline/afad_earthquake_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import auth, drive
from google.auth import default
import gspread

drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
import pandas as pd

sheet_title = "afad_earthquake_data_raw"  # Drive'daki Google Sheets dosyasının adı
csv_cache_path = "/content/drive/MyDrive/afad_earthquake_data_raw.csv"
api_url = "https://servisnet.afad.gov.tr/apigateway/deprem/apiv2/event/filter?start=1995-01-01T00:00:00&end=2026-10-06T23:59:59&minmag=2.5&format=csv&orderby=time"

try:
    # Google Sheets üzerinden okumayı dene
    worksheet = gc.open(sheet_title).sheet1
    data = worksheet.get_all_records()
    df = pd.DataFrame(data)
    print("Loaded from Google Sheets")

    # CSV olarak da Drive'a yedekle
    if not os.path.exists(csv_cache_path):
        df.to_csv(csv_cache_path, index=False)

except Exception:
    # Sheets yoksa Drive'daki CSV'yi dene, o da yoksa API'den çek
    if os.path.exists(csv_cache_path):
        print("Loaded from Google Drive (CSV)")
        df = pd.read_csv(csv_cache_path)
    else:
        print("Loaded from API URL")
        df = pd.read_csv(api_url)
        df.to_csv(csv_cache_path, index=False)

Loaded from Google Sheets


In [15]:
print(f"Shape: {df.shape}")
df.head()

Shape: (164940, 15)


,Rms,EventID,Location,Latitude,Longitude,Depth,Type,Magnitude,Country,Province,District,Neighborhood,Date,isEventUpdate,lastUpdateDate
0,0.0,243966,Sivrice (Elazığ),38.42,39.43,17.0,Md,3.2,Türkiye,Elazığ,Sivrice,Akbuğday,1/1/1995 12:48:08 AM,FALSE,
1,0.0,243967,Kartepe (Kocaeli),40.74,30.2,3.0,Md,3.1,Türkiye,Kocaeli,Kartepe,Eşme,1/1/1995 6:30:47 AM,FALSE,
2,0.0,243968,Gemlik (Bursa),40.4,29.11,10.0,Md,3.3,Türkiye,Bursa,Gemlik,Ata,1/1/1995 8:57:01 AM,FALSE,
3,0.0,243969,Gemlik (Bursa),40.41,29.1,11.0,Md,3.4,Türkiye,Bursa,Gemlik,Ata,1/2/1995 5:22:51 PM,FALSE,
4,0.0,243970,Kadirli (Osmaniye),37.55,36.19,4.0,Md,3.0,Türkiye,Osmaniye,Kadirli,Zindeğen,1/3/1995 8:23:18 AM,FALSE,
